# MaxText pipeline smoke test on TPU

Select an eight-device TPU runtime. This checks PP2 x TP4 and PP4 x TP2 with a
tiny four-layer, randomly initialized Qwen3-shaped model and synthetic tokens.
It checks physical parameter sharding, finite forward loss, gradients and one
SGD update. It does not measure pretrained quality or training throughput.
Install dependencies only in the TPU runtime. Each layout runs in a fresh process.

The setup keeps the provider JAX/Flax/libtpu versions and fails before model creation if dependency resolution cannot satisfy them.


In [ ]:
import importlib.metadata
import json
import os
from pathlib import Path
import subprocess
import sys
import tomllib

TUNIX_REPO = "https://github.com/sontungkieu/tunix.git"
TUNIX_REVISION = "codex/maxtext-pipeline-parallelism"  # Use an exact SHA for a recorded run.
ROOT = Path("/kaggle/working/tunix") if Path("/kaggle").exists() else Path("/content/tunix")
if not ROOT.exists():
    subprocess.run(["git", "clone", TUNIX_REPO, str(ROOT)], check=True, timeout=180)
subprocess.run(["git", "fetch", "origin", TUNIX_REVISION], cwd=ROOT, check=True, timeout=180)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ROOT, check=True)

# Keep the provider's existing JAX/Flax/TPU stack. Resolve MaxText and Tunix
# dependencies together, with these constraints on every resolver invocation.
RUNTIME_PACKAGES = ("jax", "jaxlib", "libtpu", "flax")
def runtime_versions():
    result = {}
    for name in RUNTIME_PACKAGES:
        try:
            result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            result[name] = None
    return result

before = runtime_versions()
assert all(before[name] for name in ("jax", "jaxlib", "flax")), before
REPORTS = Path("/kaggle/working/pr1753_results") if Path("/kaggle").exists() else ROOT / "smoke_results"
REPORTS.mkdir(exist_ok=True)
constraints = REPORTS / "provider_runtime_constraints.txt"
constraints.write_text("".join(f"{name}=={version}\n" for name, version in before.items() if version))
probe_env = {**os.environ, "JAX_PLATFORMS": "cpu", "PJRT_DEVICE": "CPU", "USE_TORCH_XLA": "0"}
subprocess.run([sys.executable, "-c", "import jax; from flax import nnx; print('Initial JAX/Flax import OK', jax.__version__)"], env=probe_env, check=True, timeout=90)
project = tomllib.loads((ROOT / "pyproject.toml").read_text())["project"]
# The preinstalled provider libtpu is intentionally kept; do not activate the
# jax[tpu] extra, which may replace it with a different TPU runtime.
dependencies = [item.replace("jax[tpu]", "jax") for item in project["dependencies"]]
subprocess.run([
    sys.executable, "-m", "pip", "install", "--constraint", str(constraints),
    "--report", str(REPORTS / "dependency_install_report.json"),
    "--requirement", str(ROOT / "requirements/maxtext_requirements.txt"),
    *dependencies,
], check=True, timeout=900)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(ROOT)], check=True, timeout=900)
after = runtime_versions()
summary = {"ok": before == after, "before": before, "after": after,
           "tunix_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()}
(REPORTS / "setup_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
assert summary["ok"], summary
subprocess.run([
    sys.executable, "-c",
    "import jax; from flax import nnx; import maxtext; from tunix.models import automodel; "
    "print('Pinned setup import OK', jax.__version__, maxtext.__version__)",
], cwd=ROOT, env={**probe_env, "PYTHONPATH": str(ROOT)}, check=True, timeout=180)
print("MAXTEXT_PIPELINE_SETUP", summary)


In [ ]:
# The configuration used inside the smoke script is equivalent to:
# layout = MaxTextPipelineConfig(pipeline_parallelism=2, tensor_parallelism=4,
#     num_layers_per_pipeline_stage=2, num_pipeline_microbatches=8,
#     pipeline_parallel_layers=4)
# mesh = layout.create_mesh()
# model, _ = AutoModel.from_pretrained("Qwen/Qwen3-1.7B", mesh=mesh,
#     model_source=ModelSource.MAXTEXT, maxtext_pipeline_config=layout, ...)

OUTPUT = ROOT / "smoke_results"
OUTPUT.mkdir(exist_ok=True)
env = {**os.environ, "JAX_PLATFORMS": "tpu", "PYTHONPATH": str(ROOT)}
for stages in (2, 4):
    subprocess.run([
        sys.executable, str(ROOT / "examples/model_load/maxtext_pipeline_smoke.py"),
        "--pipeline-stages", str(stages),
        "--output", str(OUTPUT / f"pp{stages}_tp{8 // stages}.json"),
    ], cwd=ROOT, env=env, check=True)
